# RQ2 procedure 3: from profiling traces to a validated, (m,k)-aware CBS budget

Procedure 2 (budget for a tolerance $p$ on $P(C>Q)$) **plus the (m,k)-aware budget**: the smallest $Q$ such that at most `EPS_MK` of the sliding windows of $k$ jobs contain more than $m$ missed jobs. One notebook for the three workloads: set `WORKLOAD` in the configuration cell and run top to bottom. Every section does one thing.

| # | Section | Output |
|---|---|---|
| 0 | Configuration and helpers | |
| 1 | Load the runs, simple statistics of C, R, W | tables and tail plot |
| 2 | Feasibility: is $N\cdot p \ge 10$? | which tolerances the empirical bound supports |
| 3 | Tail bound $C_p$ of every run | empirical upper bound with confidence |
| 4 | Factors: $\alpha_{drift}$, $\alpha_{cores}$, $\alpha_{platform}$ | inflation factors |
| 5 | Budgets: Route 1, Route 2, Route 3 | $Q^*$ per scenario, route, p; written to `budgets/` |
| 5.4 | **(m,k)-aware budget, combined budget** | $Q_{(m,k)}$ and $Q^*=\max(Q_p,Q_{(m,k)})$ per scenario, route, p, (m,k); written to `budgets/` |
| 6 | Offline validation on held-out runs | pass / fail per route and p |
| 6.5 | **Combined budgets on held-out stressed runs** | rate and window test per route, p, (m,k) |
| 7 | Online validation under KubeDeadline | quality metrics per cell |
| 7.1 | **(m,k) on the online runs** | predicted vs observed violating windows (overrun-caused and other), stalls |
| 8 | Recalibration | $Q_1$ and the next budgets file |
| 9 | Replay on recorded online runs | the whole loop tested on data we already have |

**Notation.** C = CPU time of a job (`cpu_ns`), R = response time (`response_ns`), W = dispatch wait (`wait_ns`), T = period = deadline, Q = budget,
p = tolerance, the admissible probability that a job's C exceeds its budget: $P(C > Q) \le p$. $C_p$ = the $(1-p)$ quantile of C (with an upper confidence bound).

In [ ]:
from pathlib import Path

# the folder RQ2 of the repository, found from wherever the kernel runs (RQ2, analysis_procedure3 or a folder around them): no paths to edit
RQ2 = next(p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "RQ2"] if (p / "analysis_procedure3").is_dir())

# ======================== EDIT HERE (the only cell to change: pick the workload) ========================
# LEVELS of each workload: k = 1 s of jobs (24 frames, 10 images, 100 audio blocks per second)
WORKLOAD = "workload3"                        # "workload1" (video decoder), "workload2" (image segmentation) or "workload3" (audio processor)

_W = {"workload1": dict(label="video decoder",      ROOT=RQ2,                  DRIFT_DIR=RQ2 / "workload1" / "alpha_drift", VALID_DIR=RQ2 / "workload1" / "validation_p2", REPLAY_DIR=RQ2 / "workload1" / "validation", PERIOD_MS=41.667,
                  LEVELS=[("relaxed", 1e-2, (22, 24)), ("strict", 1e-3, (23, 24))]),
       "workload2": dict(label="image segmentation", ROOT=RQ2 / "workload2", DRIFT_DIR=RQ2 / "workload2" / "alpha_drift", VALID_DIR=RQ2 / "workload2" / "validation_p2", REPLAY_DIR=RQ2 / "workload2" / "validation", PERIOD_MS=100.0,
                  LEVELS=[("relaxed", 1e-2, (8, 10)), ("strict", 1e-3, (9, 10))]),
       "workload3": dict(label="audio processor",    ROOT=RQ2 / "workload3", DRIFT_DIR=RQ2 / "workload3" / "alpha_drift", VALID_DIR=RQ2 / "workload3" / "validation_p2", REPLAY_DIR=RQ2 / "workload3" / "validation", PERIOD_MS=10.0,
                  LEVELS=[("relaxed", 1e-2, (95, 100)), ("strict", 1e-3, (99, 100))])}[WORKLOAD]
ROOT      = _W["ROOT"]                        # folder with results/ (profiling)
DRIFT_DIR = _W["DRIFT_DIR"]                   # folder with results/<vm>/<scenario>/... of the other VMs of the same type (alpha_drift)
VALID_DIR = _W["VALID_DIR"]                   # online validation: iter<N>/budgets.csv (what was run) and iter<N>/results/
PERIOD_MS = _W["PERIOD_MS"]                   # period = deadline T
SCENARIOS = {"single_core": ["instance0"], "multi_core": ["instance0", "instance1"]}   # scenario -> instances

CONDITIONS = ["baseline1", "cache", "memory", "baseline2"]   # profiling session (folders in ROOT/results/<scenario>/)
BASELINES, STRESS = ["baseline1", "baseline2"], ["cache", "memory"]
DRIFT_VMS = ["worker6", "worker7"]            # other VMs of the same type (DRIFT_DIR/results/<vm>/<scenario>/baseline|cache|memory)

P_LIST = [1e-1, 1e-2, 1e-3]                   # tolerances: admissible P(C > Q)
CONF   = 0.95                                 # confidence of every upper bound
NP_MIN = 10                                   # an empirical bound at p needs N*p >= NP_MIN, otherwise: fall back to Route 3 (HWM)
HWM_P  = 1e-3                                 # p at which Route 3 takes alpha_drift

ALPHA_PLATFORM = {1: 1.03, 2: 1.06, 3: 1.11}  # alpha_platform(m) measured once per VM type with the synthetic victims, m = enemy cores
M_USED = {"single_core": 3, "multi_core": 2}  # m used per scenario
ADMISSION_MAX = 0.95                          # budget Q/T above this is not admitted

L_BOOT, N_BOOT = 1000, 200                    # moving-block bootstrap: block length (jobs), replicates
GAP_JOBS = 50                                 # overruns closer than this (jobs) count as one cluster
K_LIST = [10, 100]                            # window lengths for the (m,k) metric of section 7
K_MAX = 2                                     # maximum number of recalibrations

# ---- (m,k)-aware budget and the combined budget Q* = max(Q_p, Q_(m,k)) (sections 5.4, 6.5, 7.1) ----
LEVELS = _W["LEVELS"]                         # validation_p3: (level, p, (m, k)): the rate tolerance and the window requirement of each level (k = 1 s of jobs)
MK_REQ = [mk for _, _, mk in LEVELS]           # (m, k): at least m of the k jobs meet their deadline in ANY window of k consecutive jobs (a miss = late or skipped job)
EPS_MK = 0.01                                 # tolerance of the window requirement: allowed share of the sliding windows that violate (m, k)
RESERVATION = dict(mode="cbs", phase=0.0, delta_ms=0.02)   # model of the reservation used by the replay; the values come from section 7.2 (best of the alternatives on the online runs):
#   mode "cbs": CBS wake-up rule + refill at the server deadline;  mode "periodic": refill every T at a fixed phase (fraction of T after the release instants);  delta_ms: budget cost per job beyond its C
USE_W = True                                  # the replay starts each job at release + the dispatch wait W of the same job in the profiling trace (stalls left out of the budget criterion)
STALL_FRAC = 0.5                              # a job with dispatch wait W above this share of T, and no miss in the 3 jobs before, is a platform stall (not a budget effect)

INTERFERENCES = ["none", "memory"]             # online runs per budget: without interference, with the memory enemy (add "cache" if the cache runs are done)
ITER = 0                                      # 0 = first budgets; set to 1 after recalibrating, etc.
ONLINE_ITERS = [0, 1]                         # iterations whose online runs section 7.1 compares with the (m,k) model (the ones that exist)
OUT_DIR = RQ2 / "analysis_procedure3" / "budgets"   # where the budgets of each iteration are written
BUDGETS_FILE  = OUT_DIR / f"budgets_{WORKLOAD}_iter{ITER}.csv"   # written by section 5 (the budgets to give to KubeDeadline)
MK_FILE       = OUT_DIR / f"budgets_{WORKLOAD}_mk_iter{ITER}.csv" # written by section 5.4 (the (m,k) budgets alone)
COMBINED_FILE = OUT_DIR / f"budgets_{WORKLOAD}_combined_iter{ITER}.csv"   # written by section 5.4: Q* = max(Q_p, Q_(m,k)) for every p and every (m,k)
P3_FILE       = OUT_DIR / f"budgets_{WORKLOAD}_p3_iter{ITER}.csv"         # written by section 5.5: the budgets of the LEVELS, to run online (validation_p3)
ONLINE_BUDGETS = VALID_DIR / f"iter{ITER}" / "budgets.csv"           # budgets the online runs actually used (read by section 7)
VALID_RESULTS = VALID_DIR / f"iter{ITER}" / "results"                # online runs: <scenario>/<route>/<ptag>_<none|memory|cache>/instance*.csv
REPLAY_RESULTS = _W["REPLAY_DIR"] / "results"                       # already recorded online runs of the first validation, replayed counterfactually in section 9

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binom, beta

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.float_format", "{:.3f}".format)
ALPHA = 1 - CONF

def read_run(path):
    """one CSV: warm-up jobs removed, in job order"""
    df = pd.read_csv(path, usecols=["job_id", "cpu_ns", "response_ns", "wait_ns", "deadline_met", "skipped", "warmup"])
    return df[df.warmup == 0].sort_values("job_id").reset_index(drop=True)

def cp_upper(k, n):
    """Clopper-Pearson one-sided upper bound of a probability seen k times in n trials"""
    return 1.0 if k >= n else float(beta.ppf(1 - ALPHA, k + 1, n - k))

def os_ucb(xs, p):
    """order-statistic upper bound of the (1-p) quantile: x_(j), j = smallest rank with Binom(N, 1-p).cdf(j-1) >= CONF. NaN if j > N (N*p too small)"""
    n = len(xs)
    j = int(binom.ppf(CONF, n, 1 - p)) + 1
    return xs[j - 1] if j <= n else np.nan

def boot_ucb(x, ps):
    """moving-block bootstrap upper bound of the (1-p) quantile (keeps the clustering of the jobs)"""
    n = len(x); nb = n // L_BOOT
    if nb < 2:
        return np.full(len(ps), np.nan)
    rng = np.random.default_rng(0); q = np.empty((N_BOOT, len(ps)))
    for b in range(N_BOOT):
        starts = rng.integers(0, n - L_BOOT + 1, nb)
        q[b] = np.quantile(x[(starts[:, None] + np.arange(L_BOOT)).ravel()], [1 - p for p in ps])
    return np.quantile(q, CONF, axis=0)

def bound_table(x, ps):
    """C_p of a series x (ms): point estimate, the two upper bounds, and ucb = the larger one (NaN if N*p < NP_MIN)"""
    xs = np.sort(x)
    os_ = np.array([os_ucb(xs, p) for p in ps]); bt = boot_ucb(x, ps)
    ok = np.array([len(x) * p >= NP_MIN for p in ps])
    ucb = np.where(ok & ~np.isnan(os_), np.fmax(os_, bt), np.nan)
    return pd.DataFrame(dict(p=ps, point=np.quantile(x, [1 - p for p in ps]), orderstat=os_, block_boot=bt, ucb=ucb)).set_index("p")

def exceed_test(x, Q, p):
    """overruns x > Q: rate, number of clusters, upper bound of P(C>Q) = CP upper bound of clusters/N times the mean cluster size, and ucb <= p"""
    if np.isnan(Q):
        return np.nan, 0, np.nan, False
    idx = np.flatnonzero(x > Q); n = len(x)
    if len(idx) == 0:
        return 0.0, 0, cp_upper(0, n), cp_upper(0, n) <= p
    ncl = 1 + int(np.sum(np.diff(idx) > GAP_JOBS))
    ucb = cp_upper(ncl, n) * len(idx) / ncl
    return len(idx) / n, ncl, ucb, ucb <= p

def longest_run(b):
    """longest streak of True"""
    if not b.any():
        return 0
    edges = np.flatnonzero(np.diff(np.r_[0, b.astype(int), 0]))
    return int((edges[1::2] - edges[::2]).max())

def window_max(b, k):
    """worst number of True in any window of k consecutive jobs"""
    return int(np.convolve(b.astype(int), np.ones(k, int), "valid").max()) if len(b) >= k else int(b.sum())

def ptag(p):
    return "all" if p is None else "p1e-%d" % round(-math.log10(p))

# ---- (m,k) helpers ----
def simulate_reservation(c, Q, mode="cbs", phase=0.0, delta_ms=0.0, w=None):
    """Replay of one task in a hard reservation (runtime Q, period T) fed with the C of every job in order (ms). Returns the missed jobs (late or skipped).
    The task releases job j at j*T, runs it, and (skip policy of the runner) skips the releases that pass while a job is still running.
    mode "cbs":      wake-up rule (new server instance d=t+T, q=Q if q > (d-t)*Q/T or d <= t), a job that exhausts q waits for d, then q=Q and d=d+T.
    mode "periodic": q is refilled to Q every T at the instants phase*T + i*T, whatever the task does (unused budget is lost); a job that straddles an instant gets the new budget.
    delta_ms: extra budget cost of each job beyond its C (context switch, interrupts).
    w: dispatch wait of every job (ms, release -> first instruction, exogenous: scheduling latency, platform stalls); the job starts at release + w and the budget is only consumed from then on.
    A job that fits in its budget and ends before the next refill / the next release leaves the server full and aligned, so only the other jobs are simulated (same result, much faster)."""
    T = PERIOD_MS
    n = len(c); miss = np.zeros(n, bool)
    wv = np.zeros(n) if w is None else w
    need_all = c + delta_ms
    r_all = np.arange(n) * T
    ph = phase * T if mode == "periodic" else 0.0
    if mode == "cbs":
        plain = (need_all <= Q) & (wv + need_all <= T)
    else:
        nb = ph + T * (np.floor((r_all + wv - ph) / T) + 1)               # first refill instant after the start
        start_next = r_all + T + np.r_[wv[1:], [0.0]]
        plain = (need_all <= Q) & (r_all + wv + need_all < nb) & (nb <= start_next)   # fits, ends before the refill, and the refill is over before the next job starts
    cand = np.flatnonzero(~plain).tolist()
    cl, wl, nl = c.tolist(), wv.tolist(), need_all.tolist()
    q, d = Q, ph
    free = 0                                                               # first job that can run (the ones before were skipped); (q, d) is the true state for it
    for j in cand:
        while j >= free:
            r = j * T
            if j > free:                                                   # a plain job ran just before: server full and aligned
                q = Q
                d = -1.0 if mode == "cbs" else ph + T * math.floor((r + wl[j] - ph) / T)
            if mode == "cbs":
                if d <= r or q * T > (d - r) * Q:                          # wake-up at the release
                    d, q = r + T, Q
                t = r + wl[j]
            else:
                t = r + wl[j]
                if t >= d:
                    q = Q; d += T * (int((t - d) // T) + 1)
            need = nl[j]
            while need > 1e-9:
                if q <= 1e-9:                                              # exhausted: wait for the refill
                    t = max(t, d); q = Q; d += T
                elif mode == "periodic" and d - t < min(need, q):          # a refill instant comes before the job ends
                    need -= d - t; t = d; q = Q; d += T
                else:
                    run = min(need, q); t += run; q -= run; need -= run
            miss[j] = t > r + T + 1e-9
            nxt = max(j + 1, int(math.ceil(t / T - 1e-9)))                 # the releases before the end of the job are skipped
            miss[j + 1:nxt] = True
            free = nxt
            # periodic: the next job shares the period of this one (no refill in between) -> its budget is reduced: simulate it too
            if mode == "periodic" and free < n and d > free * T + wl[free]:
                j = free
            else:
                break
    return miss

def misses_model(c, Q, **kw):
    """missed jobs of the series c under the reservation model chosen in RESERVATION (kw overrides)"""
    return simulate_reservation(c, Q, **{**RESERVATION, **kw})

def window_counts(x, k):
    """number of True in every sliding window of k consecutive jobs"""
    cs = np.r_[0, np.cumsum(x.astype(int))]
    return cs[k:] - cs[:-k]

def violation_fraction(miss, m, k, ok=None):
    """(m,k) = at least m of k deadlines met: a window violates it when it holds more than k-m misses. Returns the share of violating sliding windows. ok (per job): windows that contain a job with ok False are left out"""
    if len(miss) < k:
        return np.nan
    viol = window_counts(miss, k) > k - m                # more than k-m misses = fewer than m met
    if ok is not None:
        viol = viol[window_counts(~ok, k) == 0]
    return float(viol.mean()) if len(viol) else np.nan

def mk_budget(c, m, k, eps, grid=300, w=None):
    """smallest Q (ms) of a geometric grid between the median and the maximum of c whose violation fraction is <= eps (bisection: more budget never gives more violations).
    Jobs with C > 2T are stalls, not part of the budget criterion."""
    keep = c <= 2 * PERIOD_MS
    c, w = c[keep], (None if w is None else w[keep])
    qs = np.geomspace(np.median(c), c.max(), grid)
    lo, hi = 0, grid - 1                               # the maximum is always enough: no overrun
    while lo < hi:
        mid = (lo + hi) // 2
        if violation_fraction(misses_model(c, qs[mid], w=w), m, k) <= eps:
            hi = mid
        else:
            lo = mid + 1
    return float(qs[lo])

def stall_mask(w, bad):
    """platform stalls of a run: W (ms) above STALL_FRAC*T while none of the 3 jobs before was missed (otherwise it is the budget carry-over of an overrun).
    The next 2 jobs are marked as well: a stall also skips the following release."""
    s = (np.nan_to_num(w, nan=0.0) > STALL_FRAC * PERIOD_MS) & (np.convolve(bad.astype(int), [0, 1, 1, 1])[:len(bad)] == 0)
    return np.convolve(s.astype(int), [1, 1, 1])[:len(s)] > 0, s

def jitter(w):
    """dispatch wait without the platform stalls (W above STALL_FRAC*T): the part of W that is latency noise"""
    return np.where(w > STALL_FRAC * PERIOD_MS, 0.0, w)

def exogenous_wait(w, bad):
    """W of an online run without the wait that the reservation itself causes: after a miss within the 3 jobs before, W contains the throttling (the replay produces that itself)"""
    recent = np.convolve(bad.astype(int), [0, 1, 1, 1])[:len(bad)] > 0
    return np.where(recent, 0.0, np.nan_to_num(w, nan=0.0))

def overrun_misses(bad, over):
    """misses (late or skipped jobs) that belong to a burst of consecutive misses containing a job with C > Q: caused by an overrun, with its skip effect and the knock-on after it.
    The other misses (a burst without any overrun: stalls, late starts) are not controlled by the budget."""
    edges = np.diff(np.r_[0, bad.astype(int), 0])
    out = np.zeros(len(bad), bool)
    for s, e in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
        if over[s:e].any():
            out[s:e] = True
    return out

## 1. Load the runs and simple statistics

Runs: the **profiling session** (`baseline1 → cache → memory → baseline2`, in `ROOT/results/`) and the runs on **other VMs** of the same type (`DRIFT_DIR/results/<vm>/`, the swapped session).
Skipped jobs have no C, R or W and are left out of the statistics. Warm-up jobs are always removed.

In [ ]:
runs = []
for scen in SCENARIOS:
    for cond in CONDITIONS:
        runs.append(dict(run=f"profiling_{cond}", group="profiling", vm="profiling", scen=scen, cond=cond,
                         kind="base" if cond in BASELINES else "stress", dir=ROOT / "results" / scen / cond))
    for vm in DRIFT_VMS:
        for cond, kind in [("baseline", "base")] + [(c, "stress") for c in STRESS]:
            runs.append(dict(run=f"drift_{vm}_{cond}", group="drift", vm=vm, scen=scen, cond=cond, kind=kind,
                             dir=DRIFT_DIR / "results" / vm / scen / cond))
runs = [r for r in runs if (r["dir"] / "instance0.csv").exists()]
print("working folder :", Path.cwd())
print("ROOT             :", ROOT.resolve(), "| exists:", (ROOT / "results").exists(), "(needs ROOT/results)")
print("DRIFT_DIR        :", DRIFT_DIR.resolve(), "| exists:", (DRIFT_DIR / "results").exists(), "(needs DRIFT_DIR/results)")
print("runs found       :", len(runs), "| profiling:", sum(r["group"] == "profiling" for r in runs), "| other VMs:", sum(r["group"] == "drift" for r in runs))
if not any(r["group"] == "profiling" for r in runs):
    raise FileNotFoundError(f"no profiling runs: looked for {ROOT.resolve() / 'results' / 'single_core' / 'baseline1' / 'instance0.csv'}. The paths in the configuration cell are relative to the working folder printed above: "
                            f"check that ROOT/results holds the profiling runs (ROOT is chosen from WORKLOAD in the configuration cell).")

data, series, wseries = {}, {}, {}                        # data: all jobs of a run; series: C (ms) of the non-skipped jobs, in job order
for r in runs:
    for inst in SCENARIOS[r["scen"]]:
        f = r["dir"] / f"{inst}.csv"
        if f.exists():
            key = (r["run"], r["scen"], inst)
            data[key] = read_run(f)
            d = data[key]; series[key] = (d[d.skipped == 0].cpu_ns / 1e6).to_numpy(); wseries[key] = (d[d.skipped == 0].wait_ns / 1e6).to_numpy()

inv = pd.DataFrame([dict(run=k[0], scenario=k[1], instance=k[2], jobs=len(data[k]), skipped=int((data[k].skipped == 1).sum()),
                         late=int(((data[k].skipped == 0) & (data[k].deadline_met == 0)).sum())) for k in data])
display(inv.set_index(["scenario", "run", "instance"]))

### 1.1 Distribution of C, R and W in the profiling session (ms)
`cv` = std / mean. The stressed runs show how much the tail moves; baseline1 and baseline2 show the drift inside one VM.

In [ ]:
def describe(v):
    q = np.quantile(v, [.5, .9, .99, .999])
    return dict(n=len(v), mean=v.mean(), cv=v.std() / v.mean(), p50=q[0], p90=q[1], p99=q[2], p999=q[3], max=v.max())

COLS = dict(C="cpu_ns", R="response_ns", W="wait_ns")
rows = []
for r in runs:
    if r["group"] != "profiling":
        continue
    for inst in SCENARIOS[r["scen"]]:
        d = data[(r["run"], r["scen"], inst)]; d = d[d.skipped == 0]
        for name, col in COLS.items():
            rows.append(dict(scenario=r["scen"], instance=inst, condition=r["cond"], quantity=name, **describe(d[col] / 1e6)))
stats = pd.DataFrame(rows)
for name in COLS:
    print(f"--- {name} ---")
    display(stats[stats.quantity == name].drop(columns="quantity").set_index(["scenario", "instance", "condition"]))

In [ ]:
# survival function P(C > x) of instance0: the right end is the tail the budget has to cover
fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(6 * len(SCENARIOS), 4), squeeze=False)
for ax, scen in zip(axes[0], SCENARIOS):
    for r in runs:
        key = (r["run"], scen, "instance0")
        if r["group"] == "profiling" and r["scen"] == scen:
            x = np.sort(series[key]); ax.loglog(x, 1 - np.arange(1, len(x) + 1) / len(x) + 1 / len(x), label=r["cond"])
    for p in P_LIST:
        ax.axhline(p, color="grey", lw=.5, ls=":")
    ax.set_title(scen); ax.set_xlabel("C (ms)"); ax.set_ylabel("P(C > x)"); ax.legend()
plt.tight_layout(); plt.show()

## 2. Feasibility: is $N\cdot p \ge 10$?

The empirical bound of the $(1-p)$ quantile needs enough jobs above it. With N jobs the expected number of jobs beyond the quantile is $N\cdot p$; we ask for $N\cdot p \ge 10$ (the order-statistic bound at 95% needs $N \gtrsim 3/p$ at the very least).
For a tolerance that does not meet it, the empirical Routes 1 and 2 are **not available** and the budget falls back to **Route 3 (high-water mark)**, or more data has to be collected.
EVT is not part of the procedure: a practitioner who wants it has to check its fit separately.

In [ ]:
rows = [dict(scenario=k[1], run=k[0], instance=k[2], N=len(series[k]), **{f"N*p (p={p:g})": len(series[k]) * p for p in P_LIST}) for k in series]
feas = pd.DataFrame(rows)
display(feas.set_index(["scenario", "run", "instance"]))

feasible = {(scen, p): all(len(series[k]) * p >= NP_MIN for k in series if k[1] == scen) for scen in SCENARIOS for p in P_LIST}
print(pd.Series(feasible).unstack().rename_axis(index="scenario", columns="p").to_string())
print("\nFalse = N*p < %d in at least one run: Routes 1 and 2 fall back to Route 3 for that p." % NP_MIN)

## 3. Tail bound $C_p$ of every run

For a run with N job times $x_1,\dots,x_N$ and tolerance p, $C_p$ is the $(1-p)$ quantile, bounded from above with confidence `CONF`:

- **order statistic:** the number of jobs below the true quantile follows $\mathrm{Binomial}(N, 1-p)$, so the sorted sample element $x_{(j)}$ with $j=\min\{j: F_{Bin}(j-1)\ge CONF\}$ is an upper bound. Valid for independent jobs.
- **moving-block bootstrap:** resample blocks of `L_BOOT` consecutive jobs and take the `CONF` quantile of the bootstrap quantiles. Keeps the clustering of slow jobs.
- **$C_p$ = the larger of the two** (`ucb`). NaN when $N\cdot p < $ `NP_MIN`.

The next cells use `ucb` only.

In [ ]:
bnd = {k: bound_table(series[k], P_LIST) for k in series}
rows = [dict(scenario=k[1], instance=k[2], run=k[0], p=p, **bnd[k].loc[p].to_dict()) for k in bnd for p in P_LIST]
bt = pd.DataFrame(rows)
display(bt.pivot_table(index=["scenario", "instance", "run"], columns="p", values="ucb", sort=False))

## 4. Factors

### 4.1 $\alpha_{drift}$: variation of the bound between runs and VMs
Every baseline run of a scenario (the two profiling baselines and the baselines on the other VMs) gets its own $C_p$. With $C_p(r)$ the bound of run r:

$$\alpha_{drift}(p) = \frac{\max_r C_p(r)}{\min_r C_p(r)}$$

per instance; the deployment value is the worst instance. With n runs, a new run is covered with probability about $n/(n+1)$ (exchangeable runs), so n is reported: **few runs mean a weak guarantee**.
An episode inside one baseline is not removed: it enters the spread, which is what the factor is for.

In [ ]:
rows = []
for scen, insts in SCENARIOS.items():
    names = [r["run"] for r in runs if r["scen"] == scen and r["kind"] == "base"]
    for inst in insts:
        for p in P_LIST:
            v = np.array([bnd[(n, scen, inst)].loc[p, "ucb"] for n in names if (n, scen, inst) in bnd])
            rows.append(dict(scenario=scen, instance=inst, p=p, n_runs=len(v), alpha=v.max() / v.min()))
spread = pd.DataFrame(rows)
alpha_drift = spread.groupby(["scenario", "p"]).agg(alpha=("alpha", "max"), n_runs=("n_runs", "min"))
alpha_drift["coverage n/(n+1)"] = alpha_drift.n_runs / (alpha_drift.n_runs + 1)
display(alpha_drift)

### 4.2 $\alpha_{cores}$ (stress) and $\alpha_{platform}$
- $\alpha_{cores}(p) = C_p^{stress} / C_p^{base}$ measured on the profiling VM, one value per stressor. It tells how much the stress moves the tail (reported; Route 1 already contains it).
- $\alpha_{platform}(m)$ is the factor of the synthetic victims (measured once per VM type, `ALPHA_PLATFORM`, m = number of enemy cores). Route 2 uses it in place of the stress profiling of the task.

In [ ]:
def prof_runs(scen, kind):
    return [r["run"] for r in runs if r["scen"] == scen and r["group"] == "profiling" and r["kind"] == kind]

rows = []
for scen, insts in SCENARIOS.items():
    for inst in insts:
        for p in P_LIST:
            base = np.max([bnd[(n, scen, inst)].loc[p, "ucb"] for n in prof_runs(scen, "base")])
            for n in prof_runs(scen, "stress"):
                rows.append(dict(scenario=scen, instance=inst, p=p, stress=n.replace("profiling_", ""), alpha_cores=bnd[(n, scen, inst)].loc[p, "ucb"] / base))
ac = pd.DataFrame(rows)
display(ac.groupby(["scenario", "stress", "p"]).alpha_cores.max().unstack("p"))
print("alpha_platform used:", {s: (M_USED[s], ALPHA_PLATFORM[M_USED[s]]) for s in SCENARIOS}, "(scenario: (m, factor))")

## 5. Budgets

$C_p^{base}$ = largest bound of the two profiling baselines, $C_p^{stress}$ = largest bound over the stressed runs (cache, memory), per instance. One reservation serves all instances of a scenario, so the deployment budget $Q^*$ is the **maximum over the instances**.

| Route | Formula | Needs |
|---|---|---|
| 1 (full profiling) | $Q = C_p^{stress}\cdot\alpha_{drift}(p)$ | stress profiling of the task |
| 2 (baseline + platform factor) | $Q = C_p^{base}\cdot\alpha_{platform}(m)\cdot\alpha_{drift}(p)$ | baseline profiling only |
| 3 (high-water mark) | $Q = \max C^{stress}\cdot\alpha_{drift}(HWM_P)$, **one value for every p** | stress profiling; no tail estimate. Jobs with $C>2T$ (stalls) are left out of the maximum |

Rules applied after the formulas: (i) where $N\cdot p<10$ Route 1 and 2 are replaced by Route 3 (marked `*`); (ii) **monotone in p**: a stricter tolerance never gets a smaller budget than a looser one; (iii) admission: $Q/T \le$ `ADMISSION_MAX` for Routes 1 and 2, while **Route 3 is capped at `ADMISSION_MAX` $\cdot T$** (a high-water mark above that is useless): the cap is flagged `(capped)` and the share of stressed jobs left above it is printed.

In [ ]:
def hwm_ms(scen, inst):
    return max(series[(n, scen, inst)][series[(n, scen, inst)] <= 2 * PERIOD_MS].max() for n in prof_runs(scen, "stress"))

def finalize(raw):
    """raw: scenario, route, p, Q_raw (ms) -> fallback to Route 3 where NaN, monotone envelope in p, Q/T, admission"""
    out = []
    for (scen, route), g in raw.groupby(["scenario", "route"], sort=False):
        g = g.sort_values("p", ascending=False).copy()                    # loosest tolerance first
        h = raw[(raw.scenario == scen) & (raw.route == "hwm")].Q_raw.iloc[0]
        g["fallback"] = g.Q_raw.isna() & (route != "hwm")
        g["Q_ms"] = g.Q_raw.fillna(h).cummax()                           # a stricter p is never cheaper
        out.append(g)
    out = pd.concat(out)
    out["Q_over_T"] = out.Q_ms / PERIOD_MS
    out["admitted"] = out.Q_over_T <= ADMISSION_MAX
    return out.reset_index(drop=True)

rows = []
for scen, insts in SCENARIOS.items():
    a_h = alpha_drift.loc[(scen, HWM_P), "alpha"]; a_h = 1.0 if np.isnan(a_h) else a_h
    q3_raw = max(hwm_ms(scen, i) for i in insts) * a_h
    q3 = min(q3_raw, ADMISSION_MAX * PERIOD_MS)                  # Route 3 is only useful if admissible: cap it at the largest admissible budget
    for p in P_LIST:
        a = alpha_drift.loc[(scen, p), "alpha"]
        q1, q2 = [], []
        for inst in insts:
            base = np.max([bnd[(n, scen, inst)].loc[p, "ucb"] for n in prof_runs(scen, "base")])
            stress = np.max([bnd[(n, scen, inst)].loc[p, "ucb"] for n in prof_runs(scen, "stress")])
            q1.append(stress * a); q2.append(base * ALPHA_PLATFORM[M_USED[scen]] * a)
        rows += [dict(scenario=scen, route="route1", p=p, Q_raw=np.max(q1)), dict(scenario=scen, route="route2b", p=p, Q_raw=np.max(q2)),
                 dict(scenario=scen, route="hwm", p=p, Q_raw=q3, hwm_capped=bool(q3_raw > q3))]
raw = pd.DataFrame(rows)
raw["hwm_capped"] = raw.hwm_capped.fillna(False).astype(bool)
bud = finalize(raw)

show = bud.assign(cell=lambda x: x.Q_ms.round(3).astype(str) + " (" + x.Q_over_T.round(3).astype(str) + ")" + np.where(x.fallback, "*", "") + np.where(x.hwm_capped, " (capped)", "") + np.where(x.admitted, "", " NOT ADMITTED"))
display(show.pivot_table(index=["scenario", "route"], columns="p", values="cell", aggfunc="first"))
print("cell = Q* in ms (Q*/T);  * = fallback to Route 3 because N*p < %d;  (capped) = Route 3 cut at ADMISSION_MAX * T" % NP_MIN)
for scen in SCENARIOS:                                                            # a capped Route 3 is not a high-water mark any more: show what it leaves uncovered
    g = bud[(bud.scenario == scen) & (bud.route == "hwm") & bud.hwm_capped]
    if len(g):
        c = np.concatenate([series[(n, scen, i)] for n in prof_runs(scen, "stress") for i in SCENARIOS[scen]])
        print(f"{scen}: Route 3 capped at {g.Q_ms.iloc[0]:.2f} ms; {np.mean(c > g.Q_ms.iloc[0]):.2e} of the stressed-run jobs are still above it")

In [ ]:
fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(6 * len(SCENARIOS), 4), squeeze=False)
for ax, scen in zip(axes[0], SCENARIOS):
    for route, g in bud[bud.scenario == scen].groupby("route"):
        ax.plot(g.p, g.Q_over_T, "o-", label=route)
    ax.axhline(ADMISSION_MAX, color="red", ls=":", lw=1); ax.set_xscale("log"); ax.invert_xaxis()
    ax.set_title(scen); ax.set_xlabel("tolerance p"); ax.set_ylabel("Q* / T"); ax.legend()
plt.tight_layout(); plt.show()

# file for the validation driver: same format as validation/budgets.csv (Route 3 is one row, p = "all")
def export_budgets(table, path):
    OUT_DIR.mkdir(exist_ok=True)
    exp = table[(table.route != "hwm") | (table.p == HWM_P)].copy()
    exp["p"] = [("all" if r == "hwm" else p) for r, p in zip(exp.route, exp.p)]
    exp["runtime_us"] = (exp.Q_ms * 1000).round().astype(int); exp["period_us"] = round(PERIOD_MS * 1000)
    exp[["scenario", "route", "p", "Q_ms", "Q_over_T", "runtime_us", "period_us", "admitted"]].to_csv(path, index=False)

export_budgets(bud, BUDGETS_FILE)
print("written", BUDGETS_FILE)

### 5.4 The (m,k)-aware budget and the combined budget

The tolerance $p$ bounds how **often** a job exceeds its budget. An application can also need a **window requirement**, the **(m,k) constraint**: in any $k$ consecutive jobs **at least $m$ meet their deadline** (so at most $k-m$ miss). Overruns that come in bursts break it even when $P(C>Q)\le p$.
Under a hard CBS reservation with the skip policy an overrun costs **more than one miss** (the late job and the releases skipped while it finishes), so a window tolerates one isolated overrun only if $k-m\ge2$; with $k-m=1$ (e.g. (9,10)) the requirement means **no overrun in any window**.

**It needs its own tolerance.** "At least $m$ of any $k$ deadlines met, always" cannot be guaranteed statistically (it would push $Q$ to the maximum observed). The requirement is stated like $p$: the share of sliding windows of $k$ jobs that violate it is at most $\varepsilon$ (`EPS_MK`).

**Criterion.** For a candidate budget $Q$ the profiling trace is replayed through a model of the reservation (`simulate_reservation`, chosen in `RESERVATION`): a job that exhausts its budget is throttled until the refill and ends late, the releases that pass meanwhile are skipped, and the **state of the server after the overrun** (remaining budget, deadline or refill instant) is carried to the next jobs, which is how a job with $C\le Q$ can still be late. The violation fraction is

$$V_{m,k}(Q) = \frac{\#\{\text{windows of } k \text{ jobs with more than } k-m \text{ misses}\}}{\#\{\text{windows}\}}$$

and $Q_{(m,k)}$ is the smallest $Q$ with $V_{m,k}(Q)\le\varepsilon$, taken on **every** profiling run of the route and maximised over runs and instances:

| Route | Traces replayed | Formula |
|---|---|---|
| 1 | stressed runs (cache, memory) | $Q_{(m,k)} = \max_{runs} Q_{prof}(m,k,\varepsilon)\cdot\alpha_{drift}$ |
| 2 | baseline runs | $Q_{(m,k)} = \max_{runs} Q_{prof}(m,k,\varepsilon)\cdot\alpha_{platform}\cdot\alpha_{drift}$ |

$\alpha_{drift}$ is the **largest** of section 4.1 over the tolerances (the (m,k) requirement has no $p$). Scaling $Q$ is the same as scaling the traces: the model only depends on $C/Q$.

**Both requirements.** Each one is a constraint on the budget and both improve as $Q$ grows, so the smallest budget meeting both is

$$Q^* = \max\big(Q_p,\ Q_{(m,k)}\big)$$

with $Q_p$ the budget of section 5 for the same route and $p$. The table shows which one dominates.

**Dispatch wait W.** With `USE_W` each replayed job starts at its release plus the wait W it had in the profiling trace (scheduling latency noise; the budget is only consumed from the start). The platform stalls (W above `STALL_FRAC`$\cdot T$) are taken out of the budget search because no budget removes them; instead the **floor** table gives the share of windows that violate $(m,k)$ with an unlimited budget: a requirement whose floor is above $\varepsilon$ is not achievable on this platform, the analogue of $p_{floor}$.

**What the budget guarantees, and what it does not.** $Q^*$ is about the misses **caused by overruns** (with their skip effect): this is what the model counts and what section 7.1 checks with `obs_overrun`. The deadline requirement the application *experiences* also contains stalls and late starts; $Q$ does not control them, so they are measured and reported (`obs_total`, `obs_other`, `stalls`), like $p_{floor}$ for the rate. A stall longer than the window breaks $(m,k)$ for any budget: that is a statement about the platform. Jobs with $C>2T$ are left out of the budget criterion.
$V$ is estimated on one run with overlapping (dependent) windows: no confidence bound is attached; the variation between runs is what $\alpha_{drift}$ covers, the rest is checked online. Whether the model reproduces the knock-on misses (misses with $C\le Q$ after an overrun) depends on how the reservation really refills its budget: section 7.1 compares the alternatives on the online runs and the winner is set in `RESERVATION`.

In [ ]:
rows = []
for scen, insts in SCENARIOS.items():
    a = alpha_drift.loc[scen].alpha.max(); a = 1.0 if np.isnan(a) else a
    for m, k in MK_REQ:
        wj = (lambda key: jitter(wseries[key])) if USE_W else (lambda key: None)
        q1 = max(mk_budget(series[(n, scen, i)], m, k, EPS_MK, w=wj((n, scen, i))) for n in prof_runs(scen, "stress") for i in insts)
        q2 = max(mk_budget(series[(n, scen, i)], m, k, EPS_MK, w=wj((n, scen, i))) for n in prof_runs(scen, "base") for i in insts)
        rows += [dict(scenario=scen, route="route1", m=m, k=k, Q_prof=q1, Q_ms=q1 * a),
                 dict(scenario=scen, route="route2b", m=m, k=k, Q_prof=q2, Q_ms=q2 * ALPHA_PLATFORM[M_USED[scen]] * a)]
mk = pd.DataFrame(rows)
mk["Q_over_T"] = mk.Q_ms / PERIOD_MS
mk["admitted"] = mk.Q_over_T <= ADMISSION_MAX
display(mk.set_index(["scenario", "route", "m", "k"]))

# combined budget Q* = max(Q_p, Q_(m,k)) per scenario, route, p, (m,k); Route 3 is one budget (p = "all") compared with Route 1's Q_(m,k)
rows = []
for b in bud.itertuples():
    if b.route == "hwm" and b.p != HWM_P:
        continue
    for q in mk[(mk.scenario == b.scenario) & (mk.route == ("route1" if b.route == "hwm" else b.route))].itertuples():
        star = max(b.Q_ms, q.Q_ms)
        rows.append(dict(scenario=b.scenario, route=b.route, p="all" if b.route == "hwm" else b.p, m=q.m, k=q.k, eps=EPS_MK, Q_p_ms=b.Q_ms, Q_mk_ms=q.Q_ms, Q_ms=star,
                         dominated_by="(m,k)" if q.Q_ms > b.Q_ms else "p", Q_over_T=star / PERIOD_MS, runtime_us=int(round(star * 1000)), period_us=round(PERIOD_MS * 1000),
                         admitted=star / PERIOD_MS <= ADMISSION_MAX))
comb = pd.DataFrame(rows)
show = comb.assign(cell=lambda x: x.Q_over_T.round(3).astype(str) + " " + np.where(x.dominated_by == "p", "(p)", "(mk)") + np.where(x.admitted, "", " NOT ADMITTED"))
display(show.pivot_table(index=["scenario", "route", "p"], columns=["m", "k"], values="cell", aggfunc="first"))
print("cell = Q*/T of max(Q_p, Q_(m,k)); (p) = the overrun-rate budget dominates, (mk) = the window requirement dominates")
print(comb.groupby(["scenario", "route"]).dominated_by.value_counts().unstack(fill_value=0).to_string())

# floor of the window requirement: violating windows that NO budget removes (stalls and late starts of the profiling runs, replayed with an unlimited budget)
if USE_W:
    fl = pd.DataFrame([dict(scenario=scen, m=m, k=k, floor=max(violation_fraction(misses_model(series[(n, scen, i)], 1e6, w=wseries[(n, scen, i)]), m, k)
                                                                  for n in prof_runs(scen, "base") + prof_runs(scen, "stress") for i in insts))
                       for scen, insts in SCENARIOS.items() for m, k in MK_REQ])
    fl["achievable"] = fl.floor <= EPS_MK
    display(fl.set_index(["scenario", "m", "k"]))
    print("floor = share of windows that violate (m,k) with an unlimited budget: if it exceeds EPS_MK the requirement cannot be met on this platform whatever Q")

OUT_DIR.mkdir(exist_ok=True)
mk.assign(eps=EPS_MK, runtime_us=(mk.Q_ms * 1000).round().astype(int), period_us=round(PERIOD_MS * 1000))[
    ["scenario", "route", "m", "k", "eps", "Q_ms", "Q_over_T", "runtime_us", "period_us", "admitted"]].to_csv(MK_FILE, index=False)
comb.to_csv(COMBINED_FILE, index=False)
print("written", MK_FILE, "and", COMBINED_FILE)

In [ ]:
# how the criterion picks Q: violation fraction of the stressed profiling run as a function of the budget
fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(6 * len(SCENARIOS), 4), squeeze=False)
for ax, scen in zip(axes[0], SCENARIOS):
    c = series[(prof_runs(scen, "stress")[-1], scen, "instance0")]; c = c[c <= 2 * PERIOD_MS]
    qs = np.geomspace(np.median(c), c.max(), 60)
    for m, k in MK_REQ:
        ax.semilogy(qs / PERIOD_MS, [max(violation_fraction(misses_model(c, q), m, k), 1e-6) for q in qs], label=f"({m},{k})")
    ax.axhline(EPS_MK, color="red", ls=":", lw=1); ax.set_title(f"{scen}, {prof_runs(scen, 'stress')[-1]}")
    ax.set_xlabel("Q / T"); ax.set_ylabel("violating windows (floor 1e-6)"); ax.legend()
plt.tight_layout(); plt.show()

### 5.5 Budgets of the levels (validation_p3)

Each level pairs a rate tolerance $p$ with a window requirement $(m,k)$, $k$ = one second of jobs, $\varepsilon$ = `EPS_MK` for both. The budget of a level and route is $Q^*=\max(Q_p,Q_{(m,k)})$ (5.4). The table gives the extra budget the window requirement costs, $Q^*/Q_p-1$, and which requirement dominates. The relaxed level tolerates a single overrun in a window (two or more misses allowed), the strict one effectively none.
Routes 1 and 2 only: Route 3 is a high-water mark and has no window effect. `P3_FILE` is the list of budgets to run online.

In [ ]:
rows = []
for name, p, (m, k) in LEVELS:
    sel = comb[(comb.route != "hwm") & np.isclose(comb.p.where(comb.p != "all", np.nan).astype(float), p) & (comb.m == m) & (comb.k == k)]
    for r in sel.itertuples():
        rows.append(dict(scenario=r.scenario, route=r.route, level=name, p=p, m=m, k=k, eps=EPS_MK, Q_p_ms=r.Q_p_ms, Q_mk_ms=r.Q_mk_ms, Q_ms=r.Q_ms,
                         extra_over_Q_p=r.Q_ms / r.Q_p_ms - 1, dominated_by=r.dominated_by, Q_over_T=r.Q_over_T, runtime_us=r.runtime_us, period_us=r.period_us, admitted=r.admitted))
p3 = pd.DataFrame(rows).sort_values(["scenario", "route", "p"], ascending=[True, True, False])
display(p3.set_index(["scenario", "route", "level"]))
OUT_DIR.mkdir(exist_ok=True)
p3.to_csv(P3_FILE, index=False)
print("written", P3_FILE, "|", len(p3), "budgets,", int(p3.admitted.sum()), "admitted")

## 6. Offline validation on held-out data

A budget is **valid on a held-out run** if its overrun probability $P(C>Q)$ is at most p, judged with the upper confidence bound of `exceed_test`:
overruns closer than `GAP_JOBS` jobs form one cluster, $P_{ucb} = CP_{upper}(\#clusters, N)\cdot(\text{mean cluster size})$, pass if $P_{ucb}\le p$.

- **6.1 Held-out baselines (leave one run out).** For each baseline run h except the reference (`baseline1`): $\alpha_{drift}$ from the **other** runs, $Q = C_p(\text{reference})\cdot\alpha_{drift}^{(-h)}$, applied to h. Tests whether the spread rule predicts a new run.
- **6.2 Held-out stressed runs.** The stressed runs on the other VMs are not used by any factor. The budgets of section 5 (Route 1 and 2) are applied to them.

In [ ]:
ref = "profiling_baseline1"
rows = []
for scen, insts in SCENARIOS.items():
    names = [r["run"] for r in runs if r["scen"] == scen and r["kind"] == "base"]
    for h in names:
        if h == ref:
            continue
        for inst in insts:
            for p in P_LIST:
                c_ref = bnd[(ref, scen, inst)].loc[p, "ucb"]
                v = np.array([bnd[(n, scen, inst)].loc[p, "ucb"] for n in names if n != h])
                Q = c_ref * v.max() / v.min()                                   # before: spread of the OTHER runs
                rate, ncl, ucb, ok = exceed_test(series[(h, scen, inst)], Q, p)
                Q1 = max(Q, c_ref * alpha_drift.loc[(scen, p), "alpha"])        # after: alpha_drift of 4.1 (all runs, h included)
                rate1, ncl1, ucb1, ok1 = exceed_test(series[(h, scen, inst)], Q1, p)
                rows.append(dict(scenario=scen, held_out=h, instance=inst, p=p, Q_ms=Q, overrun_rate=rate, clusters=ncl, P_ucb=ucb, passed=ok,
                                 Q1_ms=Q1, overrun_rate1=rate1, clusters1=ncl1, P_ucb1=ucb1, passed1=ok1))
loo = pd.DataFrame(rows)
display(loo.set_index(["scenario", "held_out", "instance", "p"]))
print(loo.groupby(["scenario", "p"]).agg(tests=("passed", "size"), passed_before=("passed", "sum"), passed_after=("passed1", "sum")).to_string())
print("\n'after' uses alpha_drift WITH the held-out run: a consistency check, not an independent validation (see 6.3).")

In [ ]:
rows = []
for r in runs:
    if not (r["group"] == "drift" and r["kind"] == "stress"):
        continue
    for inst in SCENARIOS[r["scen"]]:
        key = (r["run"], r["scen"], inst)
        if key not in series:
            continue
        for route in ("route1", "route2b"):
            for p in P_LIST:
                Q = bud[(bud.scenario == r["scen"]) & (bud.route == route) & (bud.p == p)].Q_ms.iloc[0]
                rate, ncl, ucb, ok = exceed_test(series[key], Q, p)
                rows.append(dict(scenario=r["scen"], held_out=r["run"], instance=inst, route=route, p=p, Q_ms=Q, overrun_rate=rate, clusters=ncl, P_ucb=ucb, passed=ok))
held = pd.DataFrame(rows)
if len(held):
    display(held.set_index(["scenario", "held_out", "instance", "route", "p"]))
    print(held.groupby(["scenario", "route", "p"]).passed.all().unstack("p").to_string())
else:
    print("no stressed runs on the other VMs")

### 6.3 Offline recalibration of the failed baseline tests
Every (held-out run, instance, p) that failed in 6.1 becomes a drift sample:

- $\alpha_{before}$ = the spread **without** that run (the factor that failed in 6.1),
- $\alpha_{after}$ = the spread **with every run** (the value of section 4.1, used by the budgets of section 5),
- $Q_0 = C_p^{ref}\cdot\alpha_{before}$ and $Q_1 = \max(Q_0,\ C_p^{ref}\cdot\alpha_{after})$.

$Q_1$ is already re-tested on the held-out run in the table of 6.1 (`*1` columns), but that run is now part of the calibration, so this is **not** an independent validation: it only shows that the recalibrated value is consistent with the data it contains. The independent offline check of the recalibrated budgets is 6.2 (stressed runs of the other VMs), and the final one is the online run.

In [ ]:
fail = loo[~loo.passed]
recal_offline = fail.assign(alpha_before=lambda x: x.Q_ms / [bnd[(ref, r.scenario, r.instance)].loc[r.p, "ucb"] for r in x.itertuples()],
                            alpha_after=lambda x: [alpha_drift.loc[(r.scenario, r.p), "alpha"] for r in x.itertuples()],
                            Q1_over_Q0=lambda x: x.Q1_ms / x.Q_ms)
cols = ["alpha_before", "alpha_after", "Q_ms", "Q1_ms", "Q1_over_Q0", "P_ucb", "P_ucb1", "passed1"]
if len(recal_offline):
    display(recal_offline.set_index(["scenario", "held_out", "instance", "p"])[cols].rename(columns={"Q_ms": "Q0", "Q1_ms": "Q1", "passed1": "passes_after"}))
else:
    print("every held-out baseline test passed: no recalibration needed")

### If a held-out test fails
An offline failure means the held-out data contain variation the profiling did not. Before changing anything:

1. **Setup:** was the run made under the same conditions (VM type, threads, input, enemy placement)? If not it is not a valid test: exclude it and say why.
2. **Episode:** is the failure confined to a stretch of the run (a stall or burst)? With an *episode-excluded* policy it is reported as an episode with its rate; with *episode-inclusive* it is a failure.
3. **Drift larger than $\alpha_{drift}$** (the usual cause when the distribution is narrow, e.g. the audio workload): $\alpha_{drift}$ came from few runs and the held-out run is a new sample outside the observed spread.

Then recalibrate **once**: the held-out run becomes one more drift sample. Section 4 already uses every baseline run, so the budgets of section 5 are the recalibrated ones; rerun from section 4 after adding the run to `alpha_drift/results/`. Do not iterate offline repeatedly: each round uses up a held-out set. After that the **online run is the independent test**.

### 6.4 Budgets to test online

The table collects, for every budget of section 5, the offline verdicts and a status. It is the list of what to run online.

- `baseline_heldout`: the drift rule on held-out baselines after recalibration (6.1, `passed1`), for the scenario and p. It does not depend on the route. In-sample for the runs that set $\alpha_{drift}$.
- `stress_heldout`: the budget on the stressed runs of the other VMs (6.2), independent of every factor. Empty if there are none, or for Route 3.
- Status: *ready* if admitted and no offline test failed; otherwise the reason. A failed offline test does not remove the cell: it is still run online, because the online run is the final test, but it is marked so the result is not a surprise.

Two files are written in `OUT_DIR`: `BUDGETS_FILE` (driver format, unchanged columns) and the same name with `_offline` (with the verdicts).

In [ ]:
base_ok = loo.groupby(["scenario", "p"]).passed1.all()
stress_ok = held.groupby(["scenario", "route", "p"]).passed.all() if len(held) else pd.Series(dtype=bool)

final = bud.copy()
final["baseline_heldout"] = [np.nan if r.route == "hwm" else base_ok.get((r.scenario, r.p), np.nan) for r in final.itertuples()]
final["stress_heldout"] = [stress_ok.get((r.scenario, r.route, r.p), np.nan) for r in final.itertuples()]

def status(r):
    if not r.admitted:
        return "NOT ADMITTED (Q/T above the limit)"
    if r.fallback:
        return "Route 3 value (N*p too small for the empirical bound)"
    if r.hwm_capped:
        return "Route 3 capped at ADMISSION_MAX*T: not a high-water mark any more"
    if pd.notna(r.stress_heldout) and not r.stress_heldout:
        return "run online: offline stress test FAILED"
    if pd.notna(r.baseline_heldout) and not r.baseline_heldout:
        return "run online: offline drift test FAILED"
    return "ready"
final["status"] = [status(r) for r in final.itertuples()]

to_show = final[(final.route != "hwm") | (final.p == HWM_P)]                   # Route 3 is one budget for every p
display(to_show[["scenario", "route", "p", "Q_ms", "Q_over_T", "admitted", "baseline_heldout", "stress_heldout", "status"]].reset_index(drop=True))

OUT_DIR.mkdir(exist_ok=True)
export_budgets(final, BUDGETS_FILE)                                              # driver file (same Q as section 5)
off = to_show.copy(); off["p"] = [("all" if r == "hwm" else p) for r, p in zip(off.route, off.p)]
off_file = BUDGETS_FILE.with_name(BUDGETS_FILE.stem + "_offline.csv")
off[["scenario", "route", "p", "Q_ms", "Q_over_T", "admitted", "baseline_heldout", "stress_heldout", "status"]].to_csv(off_file, index=False)
n_cells = int(to_show.admitted.sum())
print(f"written {BUDGETS_FILE} and {off_file}")
print(f"online runs to do: {n_cells} admitted budgets x {len(INTERFERENCES)} ({' / '.join(INTERFERENCES)}) = {len(INTERFERENCES) * n_cells}, each over {', '.join(f'{len(v)} instance(s) ({k})' for k, v in SCENARIOS.items())}")

### 6.5 The combined budgets on held-out stressed runs

The stressed runs on the other VMs (the same ones as 6.2) are replayed with the combined budgets $Q^*=\max(Q_p,Q_{(m,k)})$ of 5.4. Both requirements are tested, each with its own tolerance:
- **rate:** $P_{ucb}(C>Q^*)\le p$ (`exceed_test`, as in 6.2; Route 3 judged at `HWM_P`),
- **window:** $V_{m,k}\le$ `EPS_MK` on the replay of the run (`misses_model`).

No factor of the procedure has seen these runs. The window test counts overruns only (stalls and carry-over are outside it, see 5.4).

In [ ]:
rows = []
for r in runs:
    if not (r["group"] == "drift" and r["kind"] == "stress"):
        continue
    for inst in SCENARIOS[r["scen"]]:
        key = (r["run"], r["scen"], inst)
        if key not in series:
            continue
        for q in comb[comb.scenario == r["scen"]].itertuples():
            p = HWM_P if q.route == "hwm" else q.p
            rate, ncl, ucb, ok_p = exceed_test(series[key], q.Q_ms, p)
            v = violation_fraction(misses_model(series[key], q.Q_ms, w=jitter(wseries[key]) if USE_W else None), q.m, q.k)
            rows.append(dict(scenario=r["scen"], held_out=r["run"], instance=inst, route=q.route, p=q.p, m=q.m, k=q.k, Q_ms=q.Q_ms, dominated_by=q.dominated_by,
                             P_ucb=ucb, passed_p=ok_p, violation_fraction=v, passed_mk=v <= EPS_MK, passed=ok_p and v <= EPS_MK))
held_mk = pd.DataFrame(rows)
if len(held_mk):
    print(held_mk.groupby(["scenario", "route", "p", "m", "k"]).agg(runs=("passed", "size"), passed_both=("passed", "sum"), passed_p=("passed_p", "sum"), passed_mk=("passed_mk", "sum"),
                                                                   worst_P_ucb=("P_ucb", "max"), worst_violation=("violation_fraction", "max")).to_string())
else:
    print("no stressed runs on the other VMs")

## 7. Online validation under KubeDeadline

The budgets of `BUDGETS_FILE` (section 5) are given to KubeDeadline and the workload runs under them (without interference, with the memory enemy and with the cache enemy). Results are read from `VALID_RESULTS/<scenario>/<route>/<ptag>_<none|memory|cache>/instance*.csv`.
If the folder does not exist yet this section prints a message and the next one is skipped. To look at older runs, point `VALID_RESULTS` and `ONLINE_BUDGETS` at them in the configuration cell.

Per run (and instance) the quality metrics are:

| Metric | Meaning |
|---|---|
| `P_over`, `P_ucb`, `passed` | observed $P(C>Q)$ (jobs with $C>Q$ / non-skipped jobs), its cluster-aware upper bound, and $P_{ucb}\le p$ |
| `obs/p` | share of the allowed overruns that was used |
| `over_budget` | $Q / \hat C_p^{val} - 1$ with $\hat C_p^{val}$ the $(1-p)$ quantile seen in this run (the budget that would have been just enough); negative = budget too small |
| `miss_rate` | $P(R>T)$ = (late + skipped) / jobs, split into `late_over` (C>Q), `late_other` (C≤Q: stalls, scheduler) and `skipped` |
| `longest`, `miss@k` | longest streak of consecutive missed jobs, and the worst number of misses in any window of k jobs |

For Route 3 (one budget for every p) the run is judged at `HWM_P`.

In [ ]:
def online_row(df, Q, p):
    ok = df[df.skipped == 0]
    c = (ok.cpu_ns / 1e6).to_numpy(); late = (ok.deadline_met == 0).to_numpy()
    rate, ncl, ucb, passed = exceed_test(c, Q, p)
    bad = ((df.skipped == 1) | (df.deadline_met == 0)).to_numpy()
    row = dict(N=len(c), P_over=rate, P_ucb=ucb, passed=passed, obs_over_p=rate / p, over_budget=Q / np.quantile(c, 1 - p) - 1,
               miss_rate=bad.mean(), late_over=int((late & (c > Q)).sum()), late_other=int((late & (c <= Q)).sum()), skipped=int((df.skipped == 1).sum()),
               longest=longest_run(bad))
    row.update({f"miss@{k}": window_max(bad, k) for k in K_LIST})
    return row, c

online, val_c = [], {}
if ONLINE_BUDGETS.exists() and VALID_RESULTS.exists():
    for _, b in pd.read_csv(ONLINE_BUDGETS, dtype={"p": str}).iterrows():
        is_hwm = b.route == "hwm"
        p = HWM_P if is_hwm else float(b.p)
        for interf in INTERFERENCES:
            d = VALID_RESULTS / b.scenario / b.route / f"{'all' if is_hwm else ptag(p)}_{interf}"
            for inst in SCENARIOS[b.scenario]:
                if (d / f"{inst}.csv").exists():
                    row, c = online_row(read_run(d / f"{inst}.csv"), b.runtime_us / 1000, p)
                    online.append(dict(scenario=b.scenario, route=b.route, p=p, interference=interf, instance=inst, Q_ms=b.runtime_us / 1000, **row))
                    val_c[(b.scenario, b.route, p, interf, inst)] = c
online = pd.DataFrame(online)
if len(online):
    display(online.set_index(["scenario", "route", "p", "interference", "instance"]))
    print("passed in every instance:")
    print(online.groupby(["scenario", "route", "p", "interference"]).passed.all().unstack("interference").to_string())
else:
    print(f"no online results yet in {VALID_RESULTS} (budgets: {ONLINE_BUDGETS})")

### 7.1 The (m,k) model against the online runs

No new runs are needed: every online run already recorded (`ONLINE_ITERS`) gives, for each requirement $(m,k)$, these numbers (all as the share of sliding windows that violate it (more than $k-m$ missed jobs)):

| Column | Meaning |
|---|---|
| `pred_profiling` | the model (`misses_model`) on the **profiling** stressed runs with the budget the online run used: what the procedure would have predicted |
| `pred_model_online` | the same model on the **C of the online run**: the model error alone, without the transfer from profiling |
| `obs_overrun` | observed, counting only the misses of bursts that contain an overrun ($C>Q$): what the budget controls |
| `obs_other` | observed, the misses of bursts without an overrun (stalls, late starts): not controlled by $Q$ |
| `obs_total` | everything the application experiences |

`ok` = `obs_overrun` $\le$ `EPS_MK` (the guaranteed part). `ok_p` = the rate test of the run ($P_{ucb}(C>Q)\le p$). `covered` = the budget used is at least $Q_{(m,k)}$ of its route (Route 3 uses Route 1's); the runs used the budgets $Q_p$ of the procedure, so `covered` means they also satisfy $Q^*=\max(Q_p,Q_{(m,k)})$.
Reading it: `pred_model_online` below `obs_overrun` = the model misses knock-on misses (carry-over of the budget after an overrun). `pred_profiling` far from `pred_model_online` = the profiling traces do not transfer (drift, episodes). The online C series has no value for skipped jobs: the model runs on all releases, with the median C where a job was skipped.

In [ ]:
_pp = {}
def pred_profiling(scen, Q, m, k):
    key = (scen, round(Q, 6), m, k)
    if key not in _pp:
        _pp[key] = max(violation_fraction(misses_model(series[(n, scen, i)], Q, w=jitter(wseries[(n, scen, i)]) if USE_W else None), m, k) for n in prof_runs(scen, "stress") for i in SCENARIOS[scen])
    return _pp[key]

rows, online_runs = [], []
for it in ONLINE_ITERS:
    bfile, rdir = VALID_DIR / f"iter{it}" / "budgets.csv", VALID_DIR / f"iter{it}" / "results"
    if not (bfile.exists() and rdir.exists()):
        continue
    for _, b in pd.read_csv(bfile, dtype={"p": str}).iterrows():
        is_hwm = b.route == "hwm"
        p = HWM_P if is_hwm else float(b.p)
        Q = b.runtime_us / 1000
        for interf in INTERFERENCES:
            d = rdir / b.scenario / b.route / f"{'all' if is_hwm else ptag(p)}_{interf}"
            for inst in SCENARIOS[b.scenario]:
                if not (d / f"{inst}.csv").exists():
                    continue
                df = read_run(d / f"{inst}.csv")
                cf = (df.cpu_ns / 1e6).to_numpy(); cf = np.where(np.isnan(cf), np.nanmedian(cf), cf)     # C of every release; the skipped ones (no C) get the median
                c = (df[df.skipped == 0].cpu_ns / 1e6).to_numpy()
                bad = ((df.skipped == 1) | (df.deadline_met == 0)).to_numpy()
                by_overrun = overrun_misses(bad, ((df.cpu_ns / 1e6) > Q).to_numpy())
                dil, raw_stall = stall_mask((df.wait_ns / 1e6).to_numpy(), bad)
                w_exo = exogenous_wait((df.wait_ns / 1e6).to_numpy(), bad)
                model_miss = misses_model(cf, Q, w=jitter(w_exo) if USE_W else None)
                ok_p = exceed_test(c, Q, p)[3]
                online_runs.append(dict(cf=cf, Q=Q, over=by_overrun, bad=bad, w=w_exo))
                for m, k in MK_REQ:
                    qmk = mk[(mk.scenario == b.scenario) & (mk.route == ("route1" if is_hwm else b.route)) & (mk.m == m) & (mk.k == k)].Q_ms.iloc[0]
                    rows.append(dict(iter=it, scenario=b.scenario, route=b.route, p=p, interference=interf, instance=inst, m=m, k=k, Q_ms=Q, Q_mk=qmk, covered=Q >= qmk, ok_p=ok_p,
                                     pred_profiling=pred_profiling(b.scenario, Q, m, k), pred_model_online=violation_fraction(model_miss, m, k),
                                     obs_overrun=violation_fraction(by_overrun, m, k), obs_other=violation_fraction(bad & ~by_overrun, m, k), obs_total=violation_fraction(bad, m, k),
                                     stalls=int((np.diff(np.r_[0, raw_stall.astype(int)]) == 1).sum())))
mk_online = pd.DataFrame(rows)
if len(mk_online) == 0:
    print("no online runs in", [str(VALID_DIR / f"iter{i}") for i in ONLINE_ITERS])
else:
    mk_online["ok"] = mk_online.obs_overrun <= EPS_MK
    mk_online["both_ok"] = mk_online.ok & mk_online.ok_p
    mk_online["total_ok"] = mk_online.obs_total <= EPS_MK
    display(mk_online.set_index(["iter", "scenario", "route", "p", "interference", "instance", "m", "k"]))

In [ ]:
if len(mk_online):
    g = mk_online.groupby(["m", "k", "covered"])
    print("--- runs by covered (budget >= Q_(m,k), i.e. >= Q* = max(Q_p, Q_(m,k))) ---")
    print(g.agg(runs=("ok", "size"), window_ok=("ok", "sum"), rate_ok=("ok_p", "sum"), both_ok=("both_ok", "sum"), application_ok=("total_ok", "sum")).to_string())
    print("window_ok = overrun-caused violations <= EPS_MK; rate_ok = P_ucb <= p; application_ok = ALL misses (stalls included) <= EPS_MK")
    # model error: the model on the C of the online run against the observed overrun-caused windows
    e = mk_online[(mk_online.obs_overrun > 0) | (mk_online.pred_model_online > 0)]
    print("\n--- model on the online C vs observed overrun-caused windows (runs where either is non-zero) ---")
    print(e.groupby(["m", "k"]).apply(lambda x: pd.Series(dict(runs=len(x), model_below_obs=(x.pred_model_online < x.obs_overrun).mean(),
          median_obs_over_model=(x.obs_overrun / x.pred_model_online.replace(0, np.nan)).median()))).to_string())
    print("\n--- profiling prediction vs model on the online C ---")
    print(mk_online.groupby(["m", "k"]).apply(lambda x: pd.Series(dict(runs=len(x), prof_below_online=(x.pred_profiling < x.pred_model_online).mean()))).to_string())
    # what the budget does not control: to be reported
    print("\n--- not controlled by the budget (stalls and late starts), per scenario and interference ---")
    s = mk_online[(mk_online.k == MK_REQ[0][1]) & (mk_online.m == MK_REQ[0][0])]
    print(s.groupby(["scenario", "interference"]).agg(runs=("stalls", "size"), runs_with_stall=("stalls", lambda x: (x > 0).sum()), max_stalls=("stalls", "max"),
          max_obs_other=("obs_other", "max"), runs_with_other_violation=("obs_other", lambda x: (x > EPS_MK).sum())).to_string())

    fig, axes = plt.subplots(1, len(MK_REQ), figsize=(5 * len(MK_REQ), 4), squeeze=False)
    for ax, (m, k) in zip(axes[0], MK_REQ):
        x = mk_online[(mk_online.m == m) & (mk_online.k == k)]
        ax.loglog(x.pred_model_online.clip(lower=1e-5), x.obs_overrun.clip(lower=1e-5), "o", ms=4, label="model on online C")
        ax.loglog(x.pred_profiling.clip(lower=1e-5), x.obs_overrun.clip(lower=1e-5), "x", ms=4, label="model on profiling")
        ax.plot([1e-5, 1], [1e-5, 1], "k:", lw=1); ax.axhline(EPS_MK, color="red", ls=":", lw=1)
        ax.set_title(f"({m},{k})"); ax.set_xlabel("predicted violating windows"); ax.set_ylabel("observed, overrun-caused (floor 1e-5)"); ax.legend()
    plt.tight_layout(); plt.show()

### 7.2 Which reservation model reproduces the knock-on misses?

The replay carries the state of the reservation from one job to the next. How it refills its budget decides whether a job with $C\le Q$ can be late after an overrun, so the alternatives are tried on the C and the budget of every online run and compared with what was observed (only the misses of bursts that contain an overrun, `obs_overrun`):

- **cbs:** Linux CBS. Wake-up rule, refill at the server deadline, which moves by $T$. With releases at multiples of $T$ the server realigns itself, so a pure CBS gives (almost) no knock-on.
- **periodic:** the budget is refilled every $T$ at a fixed phase against the release instants (phase = fraction of $T$). A job that straddles a refill instant shares two budgets, and the tail of a long job eats the head of the next one.
- **delta_ms:** extra budget cost per job beyond its `cpu_ns` (context switch, interrupts), which makes jobs with $C$ just below $Q$ late.

Two rankings: `with_W` False replays without W and is compared with the overrun-caused misses; `with_W` True starts every job at release + the **exogenous** W of the online run (W left out where one of the 3 jobs before missed, because that wait is the throttling the model must produce itself) and is compared with **all** misses, stalls included.

`count_ratio` = simulated / observed misses (1 = the model reproduces the burst size); `V_ratio(m,k)` = the same for the share of violating windows; `err` = median $|\log((sim+1)/(obs+1))|$ per run. The best model (smallest `err`) is the one to put in `RESERVATION`.

In [ ]:
MODELS = [dict(mode="cbs", phase=0.0, delta_ms=dl) for dl in (0.0, 0.01, 0.02, 0.03, 0.04, 0.06)] + \
         [dict(mode="periodic", phase=ph, delta_ms=0.03) for ph in (0.25, 0.5, 0.75)]
if len(online_runs):
    rows = []
    for with_w in (False, True):                                   # False: compare with the overrun-caused misses; True: W included, compare with ALL misses
        for model in MODELS:
            sim_n, obs_n, err = 0, 0, []
            v = {mkr: [0.0, 0.0] for mkr in MK_REQ}
            for r in online_runs:
                ref = r["bad"] if with_w else r["over"]
                sim = misses_model(r["cf"], r["Q"], w=r["w"] if with_w else None, **model)
                sim_n += sim.sum(); obs_n += ref.sum()
                if sim.sum() or ref.sum():
                    err.append(abs(np.log((sim.sum() + 1) / (ref.sum() + 1))))
                for (m, k) in MK_REQ:
                    v[(m, k)][0] += violation_fraction(sim, m, k); v[(m, k)][1] += violation_fraction(ref, m, k)
            rows.append(dict(with_W=with_w, **model, count_ratio=sim_n / max(obs_n, 1), err=np.median(err), **{f"V_ratio{mkr}": a / b if b else np.nan for mkr, (a, b) in v.items()}))
    models_cmp = pd.DataFrame(rows).sort_values(["with_W", "err"])
    display(models_cmp.reset_index(drop=True))
    for with_w, g in models_cmp.groupby("with_W"):
        print("best", "with W (vs all misses)" if with_w else "without W (vs overrun-caused misses)", g.iloc[0][["mode", "phase", "delta_ms"]].to_dict())
    print("current RESERVATION:", RESERVATION, "| USE_W:", USE_W)
else:
    print("no online runs")

## 8. Recalibration

If a Route 1 or 2 cell fails online because of **overruns** ($P_{ucb} > p$, i.e. $C>Q$ counts), recalibrate with the same estimator on the validation data:

$$Q_1 = \max\big(Q_0,\ \alpha_{drift}(p)\cdot \hat C_p^{val}\big)$$

$\hat C_p^{val}$ = the empirical upper bound (section 3) of the validation C, taken as the worst run of the cell (instances, every interference). It is **not** the maximum of the run (that would make every recalibration a high-water mark), and $\alpha_{drift}$ is kept because future deployments can drift again.

- Misses with $C\le Q$ (`late_other`) are stalls or scheduler effects: a bigger budget does not remove them, so they do not trigger recalibration.
- Skipped jobs have no C, so a far-too-small budget makes $\hat C^{val}$ an underestimate; a second round corrects it.
- **Stop rule:** at most `K_MAX` recalibrations, the next one must be validated on a **fresh** run (set `ITER = ITER + 1`). After the last one the cell falls back to Route 3; if Route 3 also fails, the tolerance is **not achievable** on this platform. Budgets above `ADMISSION_MAX` are not admitted.

In [ ]:
_ucb_cache = {}
def cval(c, p):
    """C_p upper bound of a validation series (computed once per series)"""
    if id(c) not in _ucb_cache:
        _ucb_cache[id(c)] = (c, bound_table(c, P_LIST))          # c is kept so that its id stays valid
    return _ucb_cache[id(c)][1].loc[p, "ucb"]

def recalibrate(current, cell_ok, cell_c, iteration):
    """current: final budget table; cell_ok[(scenario, route, p)] = the cell passed; cell_c[(scenario, route, p)] = list of validation C series (ms).
    Returns the table of decisions and the next budget table (None if nothing changes)."""
    rows, new, changed = [], current.copy(), False
    new["Q_raw"] = new.Q_ms                                       # start from the current final budgets
    for (scen, route, p), ok in cell_ok.items():
        if route == "hwm":
            if not ok:
                print(f"Route 3 fails for {scen}: the tolerance is not achievable on this platform (or collect more runs / use another p)")
            continue
        sel = (new.scenario == scen) & (new.route == route) & (new.p == p)
        q0 = current[sel].Q_ms.iloc[0]
        if ok:
            rows.append(dict(scenario=scen, route=route, p=p, Q0=q0, action="keep (passed)"))
            continue
        vals = [cval(c, p) for c in cell_c[(scen, route, p)]]
        chat = np.nan if np.isnan(vals).all() else np.nanmax(vals)
        a = alpha_drift.loc[(scen, p), "alpha"]
        q1 = max(q0, a * chat) if not np.isnan(chat) else np.nan
        if np.isnan(q1):
            action = "no estimate (N*p < NP_MIN): collect more validation jobs"
        elif iteration + 1 > K_MAX:
            action = "limit reached: fall back to Route 3"; new.loc[sel, "Q_raw"] = np.nan; changed = True
        elif q1 / PERIOD_MS > ADMISSION_MAX:
            action = "not admitted: Q1/T above the limit"
        else:
            action = "recalibrate"; new.loc[sel, "Q_raw"] = q1; changed = True
        rows.append(dict(scenario=scen, route=route, p=p, Q0=q0, C_val=chat, alpha_drift=a, Q1=q1, Q1_over_Q0=q1 / q0, action=action))
    nxt = finalize(new.drop(columns=["Q_ms", "Q_over_T", "admitted", "fallback"])) if changed else None
    return pd.DataFrame(rows), nxt

if len(online) == 0:
    print("nothing to recalibrate: no online results")
else:
    cell_ok = online.groupby(["scenario", "route", "p"]).passed.all().to_dict()
    cell_c = {}
    for k, c in val_c.items():
        cell_c.setdefault(k[:3], []).append(c)
    recal, nxt = recalibrate(bud, cell_ok, cell_c, ITER)
    display(recal.set_index(["scenario", "route", "p"]))
    if nxt is not None:
        f = OUT_DIR / f"budgets_{WORKLOAD}_iter{ITER + 1}.csv"
        export_budgets(nxt, f)
        print("written", f, "-> validate it on fresh runs (ITER =", ITER + 1, ")")

## 9. Replay on recorded online runs

Before new online validations, the procedure can be tested on runs that **already exist**: every recorded online run is a sample of C under deployment (a run of the workload under KubeDeadline, with or without the memory enemy). The *new* budgets are judged **counterfactually** on them: overruns are the jobs with $C > Q_{new}$, whatever budget the run was recorded with.

Protocol (the same as the real one, on recorded data): the runs of a scenario are split in two halves, **V1** and **V2**.
1. Round 0: the budgets of section 5 are tested on V1. A cell passes if **every** run passes ($P_{ucb}\le p$).
2. Failing Route 1 / 2 cells are recalibrated with V1 (section 8): $Q_1=\max(Q_0,\alpha_{drift}\cdot\hat C^{V1}_p)$.
3. $Q_1$ is tested on V2, which it has never seen (and $Q_0$ is tested on V2 too, for comparison).

Columns: `cov` = share of runs that pass. `over_budget` = $Q/\hat C_p - 1$ per run, minimum = the worst run (negative = that run needed more).

**Limit of this test:** runs recorded with a tight budget lack the skipped jobs (no C) and may have longer jobs after a throttle, so their overrun counts are *optimistic*. `max_skipped` shows how much is missing. Use the result as a fast check of the procedure, and confirm with real online runs.

In [ ]:
rep = []
if REPLAY_RESULTS.exists():
    for scen in SCENARIOS:
        for f in sorted((REPLAY_RESULTS / scen).glob("*/*/instance*.csv")):
            d = read_run(f)
            rep.append(dict(scenario=scen, run=f"{f.parent.parent.name}/{f.parent.name}", interference=next((e for e in ("memory", "cache") if f.parent.name.endswith(e)), "none"),
                            instance=f.stem, c=(d[d.skipped == 0].cpu_ns / 1e6).to_numpy(), skipped_share=(d.skipped == 1).mean()))
rep = pd.DataFrame(rep)

if len(rep) == 0:
    print(f"no recorded online runs in {REPLAY_RESULTS}: nothing to replay")
else:
    order = rep.groupby("scenario").run.transform(lambda s: s.map({r: i for i, r in enumerate(sorted(s.unique()))}))
    V1, V2 = rep[order % 2 == 0], rep[order % 2 == 1]             # alternate runs: V1 calibrates, V2 is fresh
    print("runs (run x instance):", rep.groupby("scenario").size().to_dict(), "| V1:", len(V1), "V2:", len(V2))

    def replay_eval(Q, V):
        rows = []
        for _, b in Q.iterrows():
            if b.route == "hwm" and b.p != HWM_P:                  # Route 3 is one budget, judged at HWM_P
                continue
            for _, r in V[V.scenario == b.scenario].iterrows():
                rate, ncl, ucb, ok = exceed_test(r.c, b.Q_ms, b.p)
                rows.append(dict(scenario=b.scenario, route=b.route, p=b.p, P_ucb=ucb, passed=ok, skipped_share=r.skipped_share,
                                 over_budget=b.Q_ms / np.quantile(r.c, 1 - b.p) - 1))
        return pd.DataFrame(rows)

    def per_cell(ev):
        out = ev.groupby(["scenario", "route", "p"]).agg(runs=("passed", "size"), passed=("passed", "sum"), worst_P_ucb=("P_ucb", "max"),
                                                        min_over_budget=("over_budget", "min"), median_over_budget=("over_budget", "median"), max_skipped=("skipped_share", "max"))
        out["cov"] = out.passed / out.runs
        return out

    r0 = per_cell(replay_eval(bud, V1))                            # round 0 on V1
    cell_ok = (r0.passed == r0.runs).to_dict()
    cell_c = {k: list(V1[V1.scenario == k[0]].c) for k in cell_ok}
    recal, nxt = recalibrate(bud, cell_ok, cell_c, 0)             # recalibrate with V1
    print("--- round 0: budgets of section 5 on V1 ---")
    display(r0[["runs", "passed", "cov", "worst_P_ucb", "min_over_budget", "median_over_budget", "max_skipped"]])
    print("--- recalibration with V1 ---")
    display(recal.set_index(["scenario", "route", "p"]))

    q1 = (nxt if nxt is not None else bud).set_index(["scenario", "route", "p"])
    v2_old, v2_new = per_cell(replay_eval(bud, V2)), per_cell(replay_eval(q1.reset_index(), V2))
    summ = pd.DataFrame({"Q0": bud.set_index(["scenario", "route", "p"]).Q_ms, "cov_V1(Q0)": r0["cov"], "cov_V2(Q0)": v2_old["cov"],
                         "Q1": q1.Q_ms, "cov_V2(Q1)": v2_new["cov"], "worst_P_ucb_V2(Q1)": v2_new.worst_P_ucb, "min_over_budget_V2(Q1)": v2_new.min_over_budget,
                         "Q1/T": q1.Q_over_T, "admitted": q1.admitted})
    print("--- Q0 and Q1 on the fresh half V2 ---")
    display(summ)
    print("cells that pass on V2 (every run):")
    print(pd.DataFrame({"Q0": v2_old.passed == v2_old.runs, "Q1": v2_new.passed == v2_new.runs}).groupby(["scenario", "route"]).sum().to_string())

## 10. What to report for this workload

- Section 1: C, R, W statistics and the tail plot.
- Section 2: which tolerances the empirical bound supports ($N\cdot p\ge10$), which fall back to Route 3.
- Section 4: $\alpha_{drift}$ with n runs and the coverage $n/(n+1)$, $\alpha_{cores}$.
- Section 5: the budget table with $Q^*/T$ for the three routes; 5.4: the (m,k)-aware budgets and the combined $Q^*=\max(Q_p,Q_{(m,k)})$, with which requirement dominates.
- Section 6: offline held-out result (and the cause of any failure); 6.5: the combined budgets on held-out stressed runs (rate and window test).
- Section 7: per cell the observed $P(C>Q)$ with its bound against p, `over_budget`, the miss decomposition and the bursts (`longest`, `miss@k`); 7.1: predicted against observed violating windows per (m,k) split in overrun-caused (what the budget controls) and other (stalls, late starts: reported, not budgeted), and the model error.
- Section 8: iterations needed and the budget growth $Q_1/Q_0$ per iteration.
- Section 9: how many cells pass on unseen recorded runs before and after one recalibration, and the cost ($Q_1/T$).